# Notebook 2: Potok przetwarzania wstępnego (Preprocessing Pipeline)

## Cel
Implementacja kompletnego potoku przetwarzania sygnałów EKG:
1. Ocena jakości sygnału (SNR, clipping, segmenty płaskie)
2. Usunięcie dryftu bazowego (baseline wander)
3. Filtracja pasmowoprzepustowa [0.5–40 Hz]
4. Filtr notch 50 Hz (zakłócenia sieciowe)
5. Normalizacja (Z-score, Min-Max, Robust)
6. Analiza wpływu pochodnych sygnału
7. Ekstrakcja cech statystycznych dla klasycznych modeli ML
8. Zapis przetworzonych danych

**Wymaganie:** Kontrola pośrednia nr 2 – kompletny potok + ocena jakości


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as sp_signal
from scipy.fft import rfft, rfftfreq

sys.path.insert(0, '..')
from src.data_loader import (
    load_ptbxl_metadata, build_labels, load_raw_data,
    get_train_val_test_split, SUPERCLASSES,
)
from src.preprocessing import (
    assess_signal_quality,
    bandpass_filter, notch_filter, remove_baseline_wander,
    resample_signal, normalize_signal,
    compute_derivatives, preprocess_pipeline, preprocess_batch,
)
from src.feature_extraction import (
    extract_statistical_features, extract_features_batch,
    extract_features_with_derivatives, get_feature_names,
)

plt.rcParams['figure.dpi'] = 100
sns_colors = plt.cm.tab10.colors

DATA_PATH = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
FS = 100   # częstotliwość próbkowania (niższe: 100 Hz)
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

print("Moduły załadowane pomyślnie.")


In [ ]:
# Wczytaj metadane i etykiety
Y = load_ptbxl_metadata(DATA_PATH)
Y = build_labels(Y, DATA_PATH)

train_idx, val_idx, test_idx = get_train_val_test_split(Y)
print(f"Trening: {len(train_idx):,} | Walidacja: {len(val_idx):,} | Test: {len(test_idx):,}")

# Wczytaj mały podzbiór do demonstracji (50 rekordów z każdej klasy)
demo_idx = []
for cls in SUPERCLASSES:
    cls_idx = Y[(Y.label_single == cls) & Y.index.isin(train_idx)].index[:50]
    demo_idx.extend(cls_idx)

Y_demo = Y.loc[demo_idx]
X_demo_raw = load_raw_data(Y_demo, sampling_rate=FS, data_path=DATA_PATH)
y_demo = Y_demo['label_single'].values
print(f"\nDemo: {X_demo_raw.shape} | Klasy: {np.unique(y_demo)}")


## 1. Ocena jakości sygnałów

Przed przetwarzaniem sprawdzamy jakość każdego sygnału. Automatyczna ocena
pozwala wykryć i odfiltrować sygnały niskiej jakości (artefakty, brak kontaktu
elektrody, nasycenie przetwornika).


In [ ]:
# Oblicz metryki jakości dla podzbioru demo
quality_results = []
for i, ecg in enumerate(X_demo_raw):
    q = assess_signal_quality(ecg, fs=FS)
    q['class'] = y_demo[i]
    quality_results.append(q)

df_quality = pd.DataFrame(quality_results)
print("=== Statystyki jakości sygnałów ===")
print(df_quality[['snr_db', 'clipping_ratio', 'flat_ratio', 'baseline_drift', 'quality_score']].describe().round(3))


In [ ]:
# Histogramy metryk jakości
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

metrics = [
    ('snr_db', 'SNR [dB]', 'Stosunek sygnału do szumu'),
    ('clipping_ratio', 'Clipping Ratio', 'Udział nasyconych próbek'),
    ('flat_ratio', 'Flat Ratio', 'Udział płaskich segmentów'),
    ('quality_score', 'Quality Score', 'Łączna ocena jakości [0-1]'),
]

for ax, (col, xlabel, title) in zip(axes.flatten(), metrics):
    ax.hist(df_quality[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(df_quality[col].median(), color='red', linestyle='--',
               label=f'Mediana: {df_quality[col].median():.3f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Liczba sygnałów')
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig('../results/preprocessing_quality_histograms.png', bbox_inches='tight', dpi=150)
plt.show()

low_quality = (df_quality['quality_score'] < 0.5).sum()
print(f"\nSygnały niskiej jakości (score < 0.5): {low_quality} ({100*low_quality/len(df_quality):.1f}%)")


## 2. Wizualizacja efektów filtracji

Porównanie sygnału przed i po każdym kroku przetwarzania wstępnego.


In [ ]:
# Wybierz przykładowy sygnał (odprowadzenie II – najczęściej używane)
example_ecg = X_demo_raw[0].copy()
t = np.arange(len(example_ecg)) / FS

fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)

# 1. Surowy sygnał
axes[0].plot(t, example_ecg[:, 1], 'b', linewidth=0.8)
axes[0].set_title('1. Sygnał surowy (odprowadzenie II)', fontweight='bold')
axes[0].set_ylabel('Amplituda [mV]')

# 2. Po usunięciu dryftu bazowego
ecg_no_baseline = remove_baseline_wander(example_ecg, fs=FS)
axes[1].plot(t, ecg_no_baseline[:, 1], 'g', linewidth=0.8)
axes[1].set_title('2. Po usunięciu dryftu bazowego (filtr medianowy 200ms + 600ms)', fontweight='bold')
axes[1].set_ylabel('Amplituda [mV]')

# 3. Po filtrze pasmowoprzepustowym
ecg_bp = bandpass_filter(ecg_no_baseline, fs=FS)
axes[2].plot(t, ecg_bp[:, 1], 'darkorange', linewidth=0.8)
axes[2].set_title('3. Po filtrze pasmowoprzepustowym [0.5–40 Hz] Butterworth 4. rzędu', fontweight='bold')
axes[2].set_ylabel('Amplituda [mV]')

# 4. Po filtrze notch 50 Hz
ecg_notch = notch_filter(ecg_bp, fs=FS)
axes[3].plot(t, ecg_notch[:, 1], 'purple', linewidth=0.8)
axes[3].set_title('4. Po filtrze notch 50 Hz (usunięcie zakłóceń sieciowych)', fontweight='bold')
axes[3].set_ylabel('Amplituda [mV]')

# 5. Po normalizacji Z-score
ecg_norm = normalize_signal(ecg_notch, method='zscore')
axes[4].plot(t, ecg_norm[:, 1], 'red', linewidth=0.8)
axes[4].set_title('5. Po normalizacji Z-score (μ=0, σ=1)', fontweight='bold')
axes[4].set_ylabel('Amplituda [j.u.]')
axes[4].set_xlabel('Czas [s]')

plt.tight_layout()
plt.savefig('../results/preprocessing_steps.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# Porównanie widm częstotliwościowych przed i po filtracji
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (sig, label, color) in zip(axes, [
    (example_ecg[:, 1], 'Surowy sygnał', 'blue'),
    (ecg_notch[:, 1], 'Po filtracji', 'red'),
]):
    freqs = rfftfreq(len(sig), d=1.0/FS)
    power = np.abs(rfft(sig)) ** 2
    ax.semilogy(freqs, power, color=color, alpha=0.8, linewidth=1.0)
    ax.axvline(0.5,  color='gray', linestyle=':', alpha=0.7, label='0.5 Hz (dolne odcięcie)')
    ax.axvline(40.0, color='gray', linestyle='--', alpha=0.7, label='40 Hz (górne odcięcie)')
    ax.axvline(50.0, color='orange', linestyle='-.',  alpha=0.9, label='50 Hz (zakłócenia sieciowe)')
    ax.set_title(f'Widmo mocy: {label}', fontweight='bold')
    ax.set_xlabel('Częstotliwość [Hz]')
    ax.set_ylabel('Moc [log]')
    ax.set_xlim([0, FS/2])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/preprocessing_spectra.png', bbox_inches='tight', dpi=150)
plt.show()


## 3. Porównanie metod normalizacji

In [ ]:
# Porównaj trzy metody normalizacji
methods = ['zscore', 'minmax', 'robust']
colors = ['red', 'green', 'purple']
labels_norm = ['Z-score (μ=0, σ=1)', 'Min-Max ([0,1])', 'Robust (mediana/IQR)']

fig, axes = plt.subplots(len(methods), 1, figsize=(14, 10), sharex=True)
for ax, method, color, lbl in zip(axes, methods, colors, labels_norm):
    normed = normalize_signal(ecg_bp, method=method)
    ax.plot(t, normed[:, 1], color=color, linewidth=0.8)
    ax.set_title(f'Normalizacja: {lbl}', fontweight='bold')
    ax.set_ylabel('Amplituda')
    stats_str = f'min={normed[:,1].min():.2f}, max={normed[:,1].max():.2f}, std={normed[:,1].std():.2f}'
    ax.text(0.98, 0.05, stats_str, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=9, color='gray')
axes[-1].set_xlabel('Czas [s]')

plt.tight_layout()
plt.savefig('../results/preprocessing_normalization_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Analiza pochodnych sygnału

Pochodne sygnału EKG podkreślają szybkie zmiany amplitudy, co jest pomocne
w detekcji kompleksów QRS i może poprawiać wyniki klasyfikacji.


In [ ]:
# Oblicz i zwizualizuj pochodne sygnału
ecg_processed = preprocess_pipeline(example_ecg, fs=FS)
first_deriv, second_deriv = compute_derivatives(ecg_processed)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
for ax, sig, title, color in zip(axes,
    [ecg_processed[:, 1], first_deriv[:, 1], second_deriv[:, 1]],
    ['Sygnał przetworzone (odprowadzenie II)',
     '1. pochodna sygnału (podkreśla kompleksy QRS)',
     '2. pochodna sygnału (punkty przegięcia)'],
    ['steelblue', 'darkorange', 'green']
):
    ax.plot(t, sig, color=color, linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Amplituda')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Czas [s]')

plt.tight_layout()
plt.savefig('../results/preprocessing_derivatives.png', bbox_inches='tight', dpi=150)
plt.show()


## 5. Przetwarzanie pełnego zbioru danych

In [ ]:
# Przetwórz cały zbiór demo przez potok
print("Przetwarzanie potoku dla zbioru demo...")
X_demo_processed = preprocess_batch(X_demo_raw, fs=FS, verbose=True)

print(f"\nKształt przetworzonego zbioru: {X_demo_processed.shape}")
print(f"Zakres wartości: [{X_demo_processed.min():.3f}, {X_demo_processed.max():.3f}]")
print(f"Średnia: {X_demo_processed.mean():.6f}")
print(f"Odch. std: {X_demo_processed.std():.4f}")


## 6. Ekstrakcja cech dla klasycznych modeli ML

Dla klasycznych algorytmów ML (SVM, RF, LR, KNN, GB) nie możemy bezpośrednio
używać surowych sygnałów EKG (1000×12 = 12 000 wartości). Zamiast tego
ekstrahujemy wektor **228 cech** statystycznych (19 cech × 12 odprowadzeń).


In [ ]:
# Ekstrakcja cech ze zbioru demo
print("Ekstrakcja cech statystycznych...")
X_demo_features = extract_features_batch(X_demo_processed, fs=FS, verbose=True)

print(f"\nKształt macierzy cech: {X_demo_features.shape}")
print(f"  {X_demo_features.shape[0]} próbek × {X_demo_features.shape[1]} cech")

# Nazwy cech
feature_names = get_feature_names()
print(f"\nPierwsze 5 nazw cech: {feature_names[:5]}")
print(f"Ostatnie 5 nazw cech: {feature_names[-5:]}")


In [ ]:
# Ekstrakcja cech z pochodną (456 cech = 228×2)
print("Ekstrakcja cech z pochodną...")
X_demo_features_deriv = extract_features_batch(
    X_demo_processed, fs=FS, use_derivatives=True, verbose=True
)
print(f"\nKształt z pochodnymi: {X_demo_features_deriv.shape}")
print(f"  (228 cech oryginalnych + 228 cech pochodnej = {X_demo_features_deriv.shape[1]})")


In [ ]:
# Sprawdź brakujące wartości i wartości nieskończone w cechach
print("=== Kontrola jakości cech ===")
nan_count = np.isnan(X_demo_features).sum()
inf_count = np.isinf(X_demo_features).sum()
print(f"NaN w cechach:        {nan_count}")
print(f"Inf w cechach:        {inf_count}")

if nan_count > 0 or inf_count > 0:
    X_demo_features = np.nan_to_num(X_demo_features, nan=0.0, posinf=0.0, neginf=0.0)
    print("Naprawiono: zastąpiono NaN/Inf zerami")

print(f"\nStatystyki cech:")
print(f"  Min: {X_demo_features.min():.4f}")
print(f"  Max: {X_demo_features.max():.4f}")
print(f"  Średnia: {X_demo_features.mean():.4f}")
print(f"  Odch. std: {X_demo_features.std():.4f}")


In [ ]:
# Zapis przetworzonych danych do katalogu data/processed/
os.makedirs('../data/processed', exist_ok=True)

# Zapis zbioru demo (pełny zbiór wymaga więcej czasu – patrz Notebook 3)
np.save('../data/processed/X_demo_processed.npy', X_demo_processed)
np.save('../data/processed/X_demo_features.npy', X_demo_features)
np.save('../data/processed/X_demo_features_deriv.npy', X_demo_features_deriv)
np.save('../data/processed/y_demo.npy', y_demo)

print("Zapisano pliki:")
print("  data/processed/X_demo_processed.npy    – przetworzone sygnały (demo)")
print("  data/processed/X_demo_features.npy     – cechy 228-dim (demo)")
print("  data/processed/X_demo_features_deriv.npy – cechy 456-dim z pochodnymi (demo)")
print("  data/processed/y_demo.npy              – etykiety (demo)")


In [ ]:
print("\n=== PODSUMOWANIE POTOKU PRZETWARZANIA ===")
print("Kroki potoku:")
print("  1. Resampling (jeśli potrzebny)")
print("  2. Usunięcie dryftu bazowego (filtr medianowy)")
print("  3. Filtr pasmowoprzepustowy [0.5-40 Hz] Butterworth 4. rzędu")
print("  4. Filtr notch 50 Hz")
print("  5. Normalizacja Z-score (kanał po kanale)")
print()
print("Ocena jakości: SNR, clipping, flat ratio, baseline drift")
print(f"Ekstrakcja cech: 228 cech (19 × 12 odprowadzeń)")
print(f"Z pochodnymi:    456 cech (228 + 228)")
